In [ ]:
import torch
import numpy as np

def compare_model_parameters(file1, file2):
    """
    加载两个 PyTorch checkpoint 文件，并精确对比它们的模型参数 (`state_dict`)。

    Args:
        file1 (str): 第一个 checkpoint 文件的路径。
        file2 (str): 第二个 checkpoint 文件的路径。
    """
    print(f"INFO: Comparing model parameters in '{file1}' and '{file2}'...")

    # 1. 加载两个 checkpoint 文件到 CPU
    #    使用 map_location='cpu' 避免因 GPU/CPU 设备不同导致的问题
    try:
        ckpt1 = torch.load(file1, map_location='cpu')
        ckpt2 = torch.load(file2, map_location='cpu')
    except Exception as e:
        print(f"❌ ERROR: Failed to load one of the files. Details: {e}")
        return

    # 2. 提取 state_dict
    #    兼容两种情况：checkpoint 文件直接就是 state_dict，
    #    或者是一个包含 'model_state_dict' 键的字典。
    sd1 = ckpt1.get('model_state_dict', ckpt1)
    sd2 = ckpt2.get('model_state_dict', ckpt2)

    # 3. 检查参数的键（即层名）是否完全一致
    if sd1.keys() != sd2.keys():
        print("❌ RESULT: Models have different sets of keys/layers.")
        # 可选：打印出差异的键
        keys1 = set(sd1.keys())
        keys2 = set(sd2.keys())
        print(f"Keys only in first model: {keys1 - keys2}")
        print(f"Keys only in second model: {keys2 - keys1}")
        return

    # 4. 逐一对比每个参数张量
    mismatched_layers = []
    for key in sd1.keys():
        # torch.equal() 会检查形状、数据类型和每一个元素的值是否都完全相同
        if not torch.equal(sd1[key], sd2[key]):
            mismatched_layers.append(key)
    
    # 5. 报告最终结果
    if not mismatched_layers:
        print("✅ RESULT: All model parameters are identical.")
        return True
    else:
        print(f"❌ RESULT: Models have different parameter values in the following {len(mismatched_layers)} layers:")
        for key in mismatched_layers:
            print(f"  - {key}")
        return False


def find_first_difference(file1, file2, tolerance=1e-6):
    """
    找到两个模型参数中第一个不同的位置，并提供详细的差异信息。
    
    Args:
        file1 (str): 第一个模型文件路径
        file2 (str): 第二个模型文件路径
        tolerance (float): 数值比较的容差
    
    Returns:
        dict: 包含差异信息的字典，如果没有差异则返回None
    """
    print(f"🔍 Finding first difference between '{file1}' and '{file2}'...")
    
    try:
        # 加载模型
        ckpt1 = torch.load(file1, map_location='cpu')
        ckpt2 = torch.load(file2, map_location='cpu')
        
        sd1 = ckpt1.get('model_state_dict', ckpt1)
        sd2 = ckpt2.get('model_state_dict', ckpt2)
        # 答应建表
        print(sd1.keys())
        # 检查键是否一致
        if sd1.keys() != sd2.keys():
            print("❌ Models have different parameter keys")
            return {"type": "key_mismatch", "keys1": list(sd1.keys()), "keys2": list(sd2.keys())}
        
        # 按顺序检查每个参数
        for key in sorted(sd1.keys()):
            param1 = sd1[key]
            param2 = sd2[key]
            
            # 检查形状
            if param1.shape != param2.shape:
                print(f"❌ Shape mismatch in '{key}': {param1.shape} vs {param2.shape}")
                return {
                    "type": "shape_mismatch",
                    "layer": key,
                    "shape1": param1.shape,
                    "shape2": param2.shape
                }
            
            # 检查数值差异
            diff = torch.abs(param1 - param2)
            max_diff = torch.max(diff)
            
            if max_diff > tolerance:
                # 找到第一个不同的位置
                diff_pos = torch.nonzero(diff > tolerance, as_tuple=False)
                if len(diff_pos) > 0:
                    first_pos = diff_pos[0]
                    pos_tuple = tuple(first_pos.tolist())
                    
                    print(f"❌ First difference found in '{key}' at position {pos_tuple}")
                    print(f"   Value in model1: {param1[pos_tuple]:.8f}")
                    print(f"   Value in model2: {param2[pos_tuple]:.8f}")
                    print(f"   Difference: {diff[pos_tuple]:.8f}")
                    print(f"   Max difference in this layer: {max_diff:.8f}")
                    print(f"   Layer shape: {param1.shape}")
                    print(f"   Layer statistics:")
                    print(f"     Model1 - mean: {param1.mean():.6f}, std: {param1.std():.6f}")
                    print(f"     Model2 - mean: {param2.mean():.6f}, std: {param2.std():.6f}")
                    
                    return {
                        "type": "value_mismatch",
                        "layer": key,
                        "position": pos_tuple,
                        "value1": float(param1[pos_tuple]),
                        "value2": float(param2[pos_tuple]),
                        "difference": float(diff[pos_tuple]),
                        "max_diff": float(max_diff),
                        "shape": param1.shape,
                        "stats1": {"mean": float(param1.mean()), "std": float(param1.std())},
                        "stats2": {"mean": float(param2.mean()), "std": float(param2.std())}
                    }
        
        print("✅ No differences found - models are identical!")
        return None
        
    except Exception as e:
        print(f"❌ Error during comparison: {e}")
        return {"type": "error", "message": str(e)}

def compare_models_detailed(file1, file2):
    """
    详细比较两个模型，包括统计信息和差异分析
    """
    print(f"📊 Detailed comparison of '{file1}' and '{file2}'")
    print("=" * 60)
    
    try:
        ckpt1 = torch.load(file1, map_location='cpu')
        ckpt2 = torch.load(file2, map_location='cpu')
        
        sd1 = ckpt1.get('model_state_dict', ckpt1)
        sd2 = ckpt2.get('model_state_dict', ckpt2)
        
        print(f"Model 1: {len(sd1)} parameters")
        print(f"Model 2: {len(sd2)} parameters")
        
        # 计算总参数数量
        total_params1 = sum(p.numel() for p in sd1.values())
        total_params2 = sum(p.numel() for p in sd2.values())
        print(f"Total parameters - Model1: {total_params1:,}, Model2: {total_params2:,}")
        
        # 逐层比较
        identical_layers = 0
        different_layers = 0
        
        for key in sorted(sd1.keys()):
            param1 = sd1[key]
            param2 = sd2[key]
            
            if torch.equal(param1, param2):
                identical_layers += 1
                print(f"✅ {key}: Identical (shape: {param1.shape})")
            else:
                different_layers += 1
                diff = torch.abs(param1 - param2)
                max_diff = torch.max(diff)
                mean_diff = torch.mean(diff)
                print(f"❌ {key}: Different (shape: {param1.shape})")
                print(f"   Max diff: {max_diff:.8f}, Mean diff: {mean_diff:.8f}")
                print(f"   Model1 stats: mean={param1.mean():.6f}, std={param1.std():.6f}")
                print(f"   Model2 stats: mean={param2.mean():.6f}, std={param2.std():.6f}")
        
        print(f"\nSummary: {identical_layers} identical layers, {different_layers} different layers")
        
        # 找到第一个差异
        if different_layers > 0:
            print("\n" + "=" * 60)
            find_first_difference(file1, file2)
        
    except Exception as e:
        print(f"❌ Error: {e}")

In [ ]:
# 检查input文件夹下，
# all_step的文件(all_{step}.pt)中inputs是dp_index=0(0_{step}_input.pt)和dp_index=1(1_{step}_input.pt)的文件拼起来的
# all_step的文件(all_{step}.pt)中targets是dp_index=0(0_{step}_target.pt)和dp_index=1(1_{step}_target.pt)的文件拼起来的
import os
import torch

# 获取input文件夹下的所有文件
input_files = os.listdir("input")


# 检查dp_index=0和dp_index=1的文件拼起来和all_step的文件一致
for step in range(20):
    #读取pt文件中inputs部分
    tmp=[]
    for dp_index in range(2):
        input_file = f"input/{dp_index}_{step}_input.pt"
        if not os.path.exists(input_file):
            print(f"Step {step}: {input_file} not found, skipping...")
            break
        dp_input = torch.load(input_file, map_location='cpu')["inputs"]  # 移到CPU
        tmp.append(dp_input)
    all_input = torch.cat(tmp, dim=0)
    all_input_read = torch.load(f"input/all_{step}.pt", map_location='cpu')["inputs"]  # 移到CPU
    if torch.equal(all_input, all_input_read):
        print(f"Step {step}: ✅ Inputs match")
    else:
        print(f"Step {step}: ❌ Inputs don't match")
        print(f"  Concatenated shape: {all_input.shape}, Original shape: {all_input_read.shape}")
    # 读取pt文件中targets部分
    tmp=[]
    for dp_index in range(2):
        target_file = f"input/{dp_index}_{step}_target.pt"
        if not os.path.exists(target_file):
            print(f"Step {step}: {target_file} not found, skipping...")
            break
        dp_target = torch.load(target_file, map_location='cpu')["targets"]  # 移到CPU
        tmp.append(dp_target)
    all_target = torch.cat(tmp, dim=0)
    all_target_read = torch.load(f"input/all_{step}.pt", map_location='cpu')["targets"]  # 移到CPU
    if torch.equal(all_target, all_target_read):
        print(f"Step {step}: ✅ Targets match")
    else:
        print(f"Step {step}: ❌ Targets don't match")
        print(f"  Concatenated shape: {all_target.shape}, Original shape: {all_target_read.shape}")

print("Data consistency check completed!")


Step 0: ✅ Inputs match
Step 0: ✅ Targets match
Step 1: ✅ Inputs match
Step 1: ✅ Targets match
Step 2: ✅ Inputs match
Step 2: ✅ Targets match
Step 3: ✅ Inputs match
Step 3: ✅ Targets match
Step 4: ✅ Inputs match
Step 4: ✅ Targets match
Step 5: ✅ Inputs match
Step 5: ✅ Targets match
Step 6: ✅ Inputs match
Step 6: ✅ Targets match
Step 7: ✅ Inputs match
Step 7: ✅ Targets match
Step 8: ✅ Inputs match
Step 8: ✅ Targets match
Step 9: ✅ Inputs match
Step 9: ✅ Targets match
Step 10: ✅ Inputs match
Step 10: ✅ Targets match
Step 11: ✅ Inputs match
Step 11: ✅ Targets match
Step 12: ✅ Inputs match
Step 12: ✅ Targets match
Step 13: ✅ Inputs match
Step 13: ✅ Targets match
Step 14: ✅ Inputs match
Step 14: ✅ Targets match
Step 15: ✅ Inputs match
Step 15: ✅ Targets match
Step 16: ✅ Inputs match
Step 16: ✅ Targets match
Step 17: ✅ Inputs match
Step 17: ✅ Targets match
Step 18: ✅ Inputs match
Step 18: ✅ Targets match
Step 19: ✅ Inputs match
Step 19: ✅ Targets match
Data consistency check completed!


In [ ]:
# 检查input文件夹下，
# all_step的文件(all_{step}.pt)中inputs是dp_index=0(0_{step}_input.pt)和dp_index=1(1_{step}_input.pt)的文件拼起来的
# all_step的文件(all_{step}.pt)中targets是dp_index=0(0_{step}_target.pt)和dp_index=1(1_{step}_target.pt)的文件拼起来的
import os
import torch

# 获取input文件夹下的所有文件
input_files = os.listdir("input2")


# 检查dp_index=0和dp_index=1的文件拼起来和all_step的文件一致
for step in range(20):
    #读取pt文件中inputs部分
    tmp=[]
    for dp_index in range(4):
        input_file = f"input2/{dp_index}_{step}_input.pt"
        if not os.path.exists(input_file):
            print(f"Step {step}: {input_file} not found, skipping...")
            break
        dp_input = torch.load(input_file, map_location='cpu')["inputs"]  # 移到CPU
        tmp.append(dp_input)
    all_input = torch.cat(tmp, dim=0)
    all_input_read = torch.load(f"input2/all_{step}.pt", map_location='cpu')["inputs"]  # 移到CPU
    if torch.equal(all_input, all_input_read):
        print(f"Step {step}: ✅ Inputs match")
    else:
        print(f"Step {step}: ❌ Inputs don't match")
        print(f"  Concatenated shape: {all_input.shape}, Original shape: {all_input_read.shape}")
    # 读取pt文件中targets部分
    tmp=[]
    for dp_index in range(4):
        target_file = f"input2/{dp_index}_{step}_target.pt"
        if not os.path.exists(target_file):
            print(f"Step {step}: {target_file} not found, skipping...")
            break
        dp_target = torch.load(target_file, map_location='cpu')["targets"]  # 移到CPU
        tmp.append(dp_target)
    all_target = torch.cat(tmp, dim=0)
    all_target_read = torch.load(f"input2/all_{step}.pt", map_location='cpu')["targets"]  # 移到CPU
    if torch.equal(all_target, all_target_read):
        print(f"Step {step}: ✅ Targets match")
    else:
        print(f"Step {step}: ❌ Targets don't match")
        print(f"  Concatenated shape: {all_target.shape}, Original shape: {all_target_read.shape}")

print("Data consistency check completed!")


Step 0: ✅ Inputs match
Step 0: ✅ Targets match
Step 1: ✅ Inputs match
Step 1: ✅ Targets match
Step 2: ✅ Inputs match
Step 2: ✅ Targets match
Step 3: ✅ Inputs match
Step 3: ✅ Targets match
Step 4: ✅ Inputs match
Step 4: ✅ Targets match
Step 5: ✅ Inputs match
Step 5: ✅ Targets match
Step 6: ✅ Inputs match
Step 6: ✅ Targets match
Step 7: ✅ Inputs match
Step 7: ✅ Targets match
Step 8: ✅ Inputs match
Step 8: ✅ Targets match
Step 9: ✅ Inputs match
Step 9: ✅ Targets match
Step 10: ✅ Inputs match
Step 10: ✅ Targets match
Step 11: ✅ Inputs match
Step 11: ✅ Targets match
Step 12: ✅ Inputs match
Step 12: ✅ Targets match
Step 13: ✅ Inputs match
Step 13: ✅ Targets match
Step 14: ✅ Inputs match
Step 14: ✅ Targets match
Step 15: ✅ Inputs match
Step 15: ✅ Targets match
Step 16: ✅ Inputs match
Step 16: ✅ Targets match
Step 17: ✅ Inputs match
Step 17: ✅ Targets match
Step 18: ✅ Inputs match
Step 18: ✅ Targets match
Step 19: ✅ Inputs match
Step 19: ✅ Targets match
Data consistency check completed!


In [ ]:
# check input/all_{step}.pt 和 input2/all_{step}.pt 是否一致
for step in range(20):
    input_file = f"input/all_{step}.pt"
    input2_file = f"input2/all_{step}.pt"
    if torch.equal(torch.load(input_file, map_location='cpu')["inputs"], torch.load(input2_file, map_location='cpu')["inputs"]):
        print(f"Step {step}: ✅ Inputs match")
    else:
        print(f"Step {step}: ❌ Inputs don't match")
    if torch.equal(torch.load(input_file, map_location='cpu')["targets"], torch.load(input2_file, map_location='cpu')["targets"]):
        print(f"Step {step}: ✅ Targets match")
    else:
        print(f"Step {step}: ❌ Targets don't match")


Step 0: ✅ Inputs match
Step 0: ✅ Targets match
Step 1: ✅ Inputs match
Step 1: ✅ Targets match
Step 2: ✅ Inputs match
Step 2: ✅ Targets match
Step 3: ✅ Inputs match
Step 3: ✅ Targets match
Step 4: ✅ Inputs match
Step 4: ✅ Targets match
Step 5: ✅ Inputs match
Step 5: ✅ Targets match
Step 6: ✅ Inputs match
Step 6: ✅ Targets match
Step 7: ✅ Inputs match
Step 7: ✅ Targets match
Step 8: ✅ Inputs match
Step 8: ✅ Targets match
Step 9: ✅ Inputs match
Step 9: ✅ Targets match
Step 10: ✅ Inputs match
Step 10: ✅ Targets match
Step 11: ✅ Inputs match
Step 11: ✅ Targets match
Step 12: ✅ Inputs match
Step 12: ✅ Targets match
Step 13: ✅ Inputs match
Step 13: ✅ Targets match
Step 14: ✅ Inputs match
Step 14: ✅ Targets match
Step 15: ✅ Inputs match
Step 15: ✅ Targets match
Step 16: ✅ Inputs match
Step 16: ✅ Targets match
Step 17: ✅ Inputs match
Step 17: ✅ Targets match
Step 18: ✅ Inputs match
Step 18: ✅ Targets match
Step 19: ✅ Inputs match
Step 19: ✅ Targets match


In [ ]:
# 对比results412retaingrad文件夹下和results412sum文件夹下对应文件的模型参数
for step in range(20):
    file_a = f"results412retaingrad/step_{step}_model_412_0.pt"
    file_b = f"results412sum/step_{step}_model_412_0.pt"
    if os.path.exists(file_a) and os.path.exists(file_b):
        compare_model_parameters(file_a, file_b)
    else:
        print(f"Step {step}: ❌ File not found")


INFO: Comparing model parameters in 'results412retaingrad/step_0_model_412_0.pt' and 'results412sum/step_0_model_412_0.pt'...
❌ RESULT: Models have different parameter values in the following 16 layers:
  - blocks.0.fc1.weight
  - blocks.0.fc1.bias
  - blocks.0.fc2.weight
  - blocks.0.fc2.bias
  - blocks.1.fc1.weight
  - blocks.1.fc1.bias
  - blocks.1.fc2.weight
  - blocks.1.fc2.bias
  - blocks.2.fc1.weight
  - blocks.2.fc1.bias
  - blocks.2.fc2.weight
  - blocks.2.fc2.bias
  - blocks.3.fc1.weight
  - blocks.3.fc1.bias
  - blocks.3.fc2.weight
  - blocks.3.fc2.bias
INFO: Comparing model parameters in 'results412retaingrad/step_1_model_412_0.pt' and 'results412sum/step_1_model_412_0.pt'...
❌ RESULT: Models have different parameter values in the following 16 layers:
  - blocks.0.fc1.weight
  - blocks.0.fc1.bias
  - blocks.0.fc2.weight
  - blocks.0.fc2.bias
  - blocks.1.fc1.weight
  - blocks.1.fc1.bias
  - blocks.1.fc2.weight
  - blocks.1.fc2.bias
  - blocks.2.fc1.weight
  - blocks.2.fc1.b

In [ ]:
# 判断dp后是否相同 
# 比较results412retaingrad文件夹下和results412sum文件夹下同一个dp组的0，2，4，6和1，3，5，7的模型参数
for step in range(20):
    flag = 0
    # 检查results412retaingrad中偶数编号（0,2,4,6）
    files_even = [f"results412retaingrad/step_{step}_model_412_{i}.pt" for i in range(0, 8, 2)]
    for i in range(3):
        if compare_model_parameters(files_even[i], files_even[i+1])==False:
            flag = 1
            break
    # 检查results412retaingrad中奇数编号（1,3,5,7）
    files_odd = [f"results412retaingrad/step_{step}_model_412_{i}.pt" for i in range(1, 8, 2)]
    for i in range(3):
        if compare_model_parameters(files_odd[i], files_odd[i+1])==False:
            flag = 1
            break
    # 检查results412sum中偶数编号（0,2,4,6）
    files_even_sum = [f"results412sum/step_{step}_model_412_{i}.pt" for i in range(0, 8, 2)]
    for i in range(3):
        if compare_model_parameters(files_even_sum[i], files_even_sum[i+1])==False:
            flag = 1
            break
    # 检查results412sum中奇数编号（1,3,5,7）
    files_odd_sum = [f"results412sum/step_{step}_model_412_{i}.pt" for i in range(1, 8, 2)]
    for i in range(3):
        if compare_model_parameters(files_odd_sum[i], files_odd_sum[i+1])==False:
            flag = 1
            break
    if flag == 0:
        print(f"Step {step}: ✅ All dp groups match")
    else:
        print(f"Step {step}: ❌ Some dp groups don't match")

INFO: Comparing model parameters in 'results412retaingrad/step_0_model_412_0.pt' and 'results412retaingrad/step_0_model_412_2.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412retaingrad/step_0_model_412_2.pt' and 'results412retaingrad/step_0_model_412_4.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412retaingrad/step_0_model_412_4.pt' and 'results412retaingrad/step_0_model_412_6.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412retaingrad/step_0_model_412_1.pt' and 'results412retaingrad/step_0_model_412_3.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412retaingrad/step_0_model_412_3.pt' and 'results412retaingrad/step_0_model_412_5.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412retaingrad/step_0_model_412_5.pt' and 'results412retaingra

In [ ]:
# 比较init_model/init_model_retaingrad_{rank}.pt 和 init_model/init_model_sum_{rank}.pt是否相同，以及同一个dp组 0，2，4，6/1，3，5，7是否相同
import os

# 比较init_model/init_model_retaingrad_{rank}.pt 和 init_model/init_model_sum_{rank}.pt是否相同
def compare_init_models():
    all_match = True
    for rank in range(8):
        file_retain = f"init_model/init_model_retaingrad_{rank}.pt"
        file_sum = f"init_model/init_model_sum_{rank}.pt"
        compare_model_parameters(file_retain, file_sum)

# 检查同一个dp组 0,2,4,6 和 1,3,5,7 是否相同
def compare_init_models_within_dp():
    # 偶数组
    even_ranks = [0,2,4,6]
    odd_ranks = [1,3,5,7]
    def check_group(ranks, label):
        files = [f"init_model/init_model_retaingrad_{r}.pt" for r in ranks]
        for i in range(len(files)-1):
            compare_model_parameters(files[i], files[i+1])
        files_sum = [f"init_model/init_model_sum_{r}.pt" for r in ranks]
        for i in range(len(files_sum)-1):
            compare_model_parameters(files_sum[i], files_sum[i+1])
        return True

    even_ok = check_group(even_ranks, "Even (0,2,4,6)")
    odd_ok = check_group(odd_ranks, "Odd (1,3,5,7)")
    if even_ok and odd_ok:
        print("✅ All dp groups in init_model match within group")
    else:
        print("❌ Some dp groups in init_model do not match within group")

# 执行比较
compare_init_models()
compare_init_models_within_dp()
file_a="init_model/init_model_retaingrad_0.pt"
file_b="init_model/init_model_retaingrad_1.pt"
compare_model_parameters(file_a, file_b)


INFO: Comparing model parameters in 'init_model/init_model_retaingrad_0.pt' and 'init_model/init_model_sum_0.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'init_model/init_model_retaingrad_1.pt' and 'init_model/init_model_sum_1.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'init_model/init_model_retaingrad_2.pt' and 'init_model/init_model_sum_2.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'init_model/init_model_retaingrad_3.pt' and 'init_model/init_model_sum_3.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'init_model/init_model_retaingrad_4.pt' and 'init_model/init_model_sum_4.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'init_model/init_model_retaingrad_5.pt' and 'init_model/init_model_sum_5.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parame

In [ ]:
# 比较results412_1sum/step_0_model_0.pt和results412_1/step_0_model_0.pt的模型参数具体内容
all_same = True
for step in range(10):
    for rank in range(2):
        file_a = f"results412_1sum/step_{step}_model_{rank}.pt"
        file_b = f"results412_1/step_{step}_model_{rank}.pt"
        if compare_model_parameters(file_a, file_b)==False:
            all_same = False
            break
if all_same:
    print("✅ All models are identical")
else:
    print("❌ Some models are different")
# # 读取pt文件
# model_a = torch.load(file_a, map_location='cpu')
# model_b = torch.load(file_b, map_location='cpu')
# # 打印model_a和model_b的模型参数具体内容
# print(model_a)

INFO: Comparing model parameters in 'results412_1sum/step_0_model_0.pt' and 'results412_1/step_0_model_0.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412_1sum/step_0_model_1.pt' and 'results412_1/step_0_model_1.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412_1sum/step_1_model_0.pt' and 'results412_1/step_1_model_0.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412_1sum/step_1_model_1.pt' and 'results412_1/step_1_model_1.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412_1sum/step_2_model_0.pt' and 'results412_1/step_2_model_0.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412_1sum/step_2_model_1.pt' and 'results412_1/step_2_model_1.pt'...
✅ RESULT: All model parameters are identical.
INFO: Comparing model parameters in 'results412_1sum

In [ ]:
# 比较results412_4sum/step_0_model_0.pt和results412_4/step_0_model_0.pt的模型参数具体内容
all_same = True
for step in range(10):
    for rank in range(2):
        file_a = f"results412_4sum/step_{step}_model_{rank}.pt"
        file_b = f"results412_4/step_{step}_model_{rank}.pt"
        if compare_model_parameters(file_a, file_b)==False:
            all_same = False
            break
if all_same:
    print("✅ All models are identical")
else:
    print("❌ Some models are different")

INFO: Comparing model parameters in 'results412_4sum/step_0_model_0.pt' and 'results412_4/step_0_model_0.pt'...
❌ RESULT: Models have different parameter values in the following 16 layers:
  - blocks.0.fc1.weight
  - blocks.0.fc1.bias
  - blocks.0.fc2.weight
  - blocks.0.fc2.bias
  - blocks.1.fc1.weight
  - blocks.1.fc1.bias
  - blocks.1.fc2.weight
  - blocks.1.fc2.bias
  - blocks.2.fc1.weight
  - blocks.2.fc1.bias
  - blocks.2.fc2.weight
  - blocks.2.fc2.bias
  - blocks.3.fc1.weight
  - blocks.3.fc1.bias
  - blocks.3.fc2.weight
  - blocks.3.fc2.bias
INFO: Comparing model parameters in 'results412_4sum/step_1_model_0.pt' and 'results412_4/step_1_model_0.pt'...
❌ RESULT: Models have different parameter values in the following 16 layers:
  - blocks.0.fc1.weight
  - blocks.0.fc1.bias
  - blocks.0.fc2.weight
  - blocks.0.fc2.bias
  - blocks.1.fc1.weight
  - blocks.1.fc1.bias
  - blocks.1.fc2.weight
  - blocks.1.fc2.bias
  - blocks.2.fc1.weight
  - blocks.2.fc1.bias
  - blocks.2.fc2.weight


In [ ]:
# 比较results412_1sum/step_0_model_0.pt和results412_4sum/step_0_model_0.pt的模型参数具体内容
all_same = True
for step in range(20):
    for rank in range(2):
        file_a = f"results412_1sum/step_{step}_model_{rank}.pt"
        file_b = f"results412_4sum/step_{step}_model_{rank}.pt"
        if compare_model_parameters(file_a, file_b)==False:
            all_same = False
            break
if all_same:
    print("✅ All models are identical")
else:
    print("❌ Some models are different")

INFO: Comparing model parameters in 'results412_1sum/step_0_model_0.pt' and 'results412_4sum/step_0_model_0.pt'...
❌ RESULT: Models have different parameter values in the following 16 layers:
  - blocks.0.fc1.weight
  - blocks.0.fc1.bias
  - blocks.0.fc2.weight
  - blocks.0.fc2.bias
  - blocks.1.fc1.weight
  - blocks.1.fc1.bias
  - blocks.1.fc2.weight
  - blocks.1.fc2.bias
  - blocks.2.fc1.weight
  - blocks.2.fc1.bias
  - blocks.2.fc2.weight
  - blocks.2.fc2.bias
  - blocks.3.fc1.weight
  - blocks.3.fc1.bias
  - blocks.3.fc2.weight
  - blocks.3.fc2.bias
INFO: Comparing model parameters in 'results412_1sum/step_1_model_0.pt' and 'results412_4sum/step_1_model_0.pt'...
❌ RESULT: Models have different parameter values in the following 16 layers:
  - blocks.0.fc1.weight
  - blocks.0.fc1.bias
  - blocks.0.fc2.weight
  - blocks.0.fc2.bias
  - blocks.1.fc1.weight
  - blocks.1.fc1.bias
  - blocks.1.fc2.weight
  - blocks.1.fc2.bias
  - blocks.2.fc1.weight
  - blocks.2.fc1.bias
  - blocks.2.fc2.w

In [ ]:
# 比较results412_1/step_0_model_0.pt和results412_4/step_0_model_0.pt的模型参数具体内容
all_same = True
for step in range(20):
    for rank in range(2):
        file_a = f"results412_1/step_{step}_model_{rank}.pt"
        file_b = f"results412_4/step_{step}_model_{rank}.pt"
        if compare_model_parameters(file_a, file_b)==False:
            all_same = False
            break
if all_same:
    print("✅ All models are identical")
else:
    print("❌ Some models are different")

INFO: Comparing model parameters in 'results412_1/step_0_model_0.pt' and 'results412_4/step_0_model_0.pt'...
❌ RESULT: Models have different parameter values in the following 16 layers:
  - blocks.0.fc1.weight
  - blocks.0.fc1.bias
  - blocks.0.fc2.weight
  - blocks.0.fc2.bias
  - blocks.1.fc1.weight
  - blocks.1.fc1.bias
  - blocks.1.fc2.weight
  - blocks.1.fc2.bias
  - blocks.2.fc1.weight
  - blocks.2.fc1.bias
  - blocks.2.fc2.weight
  - blocks.2.fc2.bias
  - blocks.3.fc1.weight
  - blocks.3.fc1.bias
  - blocks.3.fc2.weight
  - blocks.3.fc2.bias
INFO: Comparing model parameters in 'results412_1/step_1_model_0.pt' and 'results412_4/step_1_model_0.pt'...
❌ RESULT: Models have different parameter values in the following 16 layers:
  - blocks.0.fc1.weight
  - blocks.0.fc1.bias
  - blocks.0.fc2.weight
  - blocks.0.fc2.bias
  - blocks.1.fc1.weight
  - blocks.1.fc1.bias
  - blocks.1.fc2.weight
  - blocks.1.fc2.bias
  - blocks.2.fc1.weight
  - blocks.2.fc1.bias
  - blocks.2.fc2.weight
  - bl

In [ ]:
# 求出 loss_plots_412_4 中 loss_history_rank_{rank}.txt 和 loss_plots_sum_412_4 中 loss_history_rank_{rank}.txt 差的平均值
import os
import numpy as np
for rank in range(8):
    file_a = f"loss_plots_sum_412_4/loss_history_rank_{rank}.txt"
    file_b = f"loss_plots_412_4/loss_history_rank_{rank}.txt"
    if os.path.exists(file_a) and os.path.exists(file_b):
        losses_a = np.loadtxt(file_a, usecols=1)
        losses_b = np.loadtxt(file_b, usecols=1)
        if len(losses_a) == len(losses_b):
            diff = np.abs(losses_a - losses_b)
            mean_diff = np.mean(diff)
            print(f"Rank {rank}: Mean absolute difference in loss history: {mean_diff:.8f}")
        else:
            print(f"Rank {rank}: ❌ Loss histories have different lengths")
    else:
        print(f"Rank {rank}: ❌ One of the files does not exist")

Rank 0: Mean absolute difference in loss history: 0.44155273
Rank 1: Mean absolute difference in loss history: 0.44145508
Rank 2: Mean absolute difference in loss history: 0.49050293
Rank 3: Mean absolute difference in loss history: 0.49013672
Rank 4: Mean absolute difference in loss history: 0.44121094
Rank 5: Mean absolute difference in loss history: 0.44101563
Rank 6: Mean absolute difference in loss history: 0.41489258
Rank 7: Mean absolute difference in loss history: 0.41508789


In [ ]:
# 打印step_0_loss_0_rank0_detail_0.1494140625.pt所有内容
import os
import torch
loss_detail_file = "loss_plots_sum_412_4/step_0_loss_0_rank0_detail_0.1494140625.pt"
if os.path.exists(loss_detail_file):
    loss_data = torch.load(loss_detail_file, map_location='cpu')
    print(loss_data)


{'loss': tensor(0.1494, dtype=torch.bfloat16, requires_grad=True)}


In [ ]:
# 检验同一个目录下input_dp1_pp1_tp1_mb4中0_step_input.pt
# input_dp4_pp1_tp1_mb4中0_step_input.pt 1_step_input.pt 2_step_input.pt 3_step_input.pt 拼起来是否相同
import os
import torch
# 加载input_dp1_pp1_tp1_mb4中的0_step_input.pt
all_same=1
for step in range(10000):
    input_file_1 = f"input_dp1_pp1_tp1_mb4/{step}_0_step_input.pt"
    data_1= torch.load(input_file_1, map_location='cpu')["inputs"]
    input_file_4_list = [f"input_dp4_pp1_tp1_mb4/{i}_{step}_input.pt" for i in range(4)]
    data_4_list = [torch.load(f, map_location='cpu')["inputs"] for f in input_file_4_list]
    data_4 = torch.cat(data_4_list, dim=0)
    if not torch.equal(data_1, data_4):
        all_same=0
        print(f"Step {step}: ❌ Inputs don't match")
        print(f"  input_dp1 shape: {data_1.shape}, input_dp4 shape: {data_4.shape}")
if all_same:
    print("✅ All inputs match")
all_same=1
for step in range(10000):
    input_file_1 = f"input_dp1_pp1_tp1_mb4/{step}_0_step_target.pt"
    data_1= torch.load(input_file_1, map_location='cpu')["target"]
    input_file_4_list = [f"input_dp4_pp1_tp1_mb4/{i}_{step}_target.pt" for i in range(4)]
    data_4_list = [torch.load(f, map_location='cpu')["target"] for f in input_file_4_list]
    data_4 = torch.cat(data_4_list, dim=0)
    if not torch.equal(data_1, data_4):
        all_same=0
        print(f"Step {step}: ❌ target don't match")
        print(f"  target_dp1 shape: {data_1.shape}, target_dp4 shape: {data_4.shape}")
if all_same:
    print("✅ All targets match")